# Tile Stitch Artefact Investigation
## Two independent problems confirmed by analysis:

### Problem 1 — Scan-field vignetting (~18% intensity drop at tile edges)
The confocal laser rolls off at the edge of each scan field. Because adjacent
column-tiles overlap by ~52px, the overlap zone contains only edge pixels from
both tiles → systematically dimmer → darker horizontal bands → slightly noisier
tau in those bands.

### Problem 2 — Y registration drift (~17–28px per column-tile boundary)
The Leica stage Y-encoder accumulates a small error across the X-scan direction.
Each column of tiles is offset ~20px lower than the previous column. This is what
causes structures to look "a few pixels too high" at column boundaries — the tissue
edge steps noticeably between adjacent tile columns.

## Three fixes to evaluate:
| Option | Fixes | Approach |
|--------|-------|----------|
| **A** | Vignetting | Flat-field correction (divide each tile by smoothed background) |
| **B** | Vignetting | Cosine taper weight (down-weight edge pixels in assembly) |
| **C** | Y drift | Phase-correlation registration (shift each tile to best alignment) |
| **D** | Both | Taper + phase-correlation combined |


In [3]:
from pathlib import Path
import numpy as np
import tifffile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, median_filter
from scipy.signal import correlate2d
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
# These should be the assembled canvas .npy files from a previous per-tile fit
# OR load from the TIFFs directly for diagnosis

CANVAS_DIR   = Path("Results/R_2/")    # directory with *_intensity.npy, *_tau_mean_amp.npy etc
ROI_NAME     = "R_2"

# If .npy files not available, load from TIFFs (8-bit demo mode)
USE_TIFF_FALLBACK = True
INTENSITY_TIFF = Path("Results/R_2/R_2_intensity.tif")
TAU_TIFF       = Path("Results/R_2/R_2_tau_intensity_weighted.tif")

# Tile geometry (read from acquisition — matches this dataset)
TILE_SIZE_PX   = 512    # pixels per tile side (pre-rotation)
TILE_PITCH_ROW = 677    # row pitch after 90°CW rotation (= original Y pitch)
TILE_PITCH_COL = 460    # col pitch after 90°CW rotation (= original X pitch)
TILE_OVERLAP   = TILE_SIZE_PX - TILE_PITCH_COL   # 52px overlap in col direction

# Output directory for corrected images
OUTPUT_DIR = Path("output/stitch_correction_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Tile size={TILE_SIZE_PX}px  row_pitch={TILE_PITCH_ROW}px  "
      f"col_pitch={TILE_PITCH_COL}px  overlap={TILE_OVERLAP}px")


Tile size=512px  row_pitch=677px  col_pitch=460px  overlap=52px


In [4]:
if USE_TIFF_FALLBACK:
    intensity = tifffile.imread(str(INTENSITY_TIFF)).astype(np.float32)
    tau_raw   = tifffile.imread(str(TAU_TIFF)).astype(np.float32)
    # Scale tau back to ns (assuming 0-5ns range, 0-255 scale for 8bit demo)
    tau = tau_raw / 255.0 * 5.0
    print(f"Loaded from TIFFs (8-bit fallback)")
else:
    intensity = np.load(str(CANVAS_DIR / f"{ROI_NAME}_intensity.npy"))
    tau       = np.load(str(CANVAS_DIR / f"{ROI_NAME}_tau_mean_amp.npy"))
    print(f"Loaded from canvas .npy files")

H, W = intensity.shape
print(f"Canvas: {H}×{W} px")
print(f"Intensity: min={intensity[intensity>0].min():.1f}  "
      f"max={intensity.max():.1f}  "
      f"mean={intensity[intensity>0].mean():.1f}")
print(f"Tau (ns): min={np.nanmin(tau[tau>0]):.3f}  "
      f"max={np.nanmax(tau):.3f}  "
      f"mean={np.nanmean(tau[tau>0]):.3f}")

# Tile column boundary positions (overlap zone START columns)
col_boundaries = list(range(TILE_PITCH_COL, W, TILE_PITCH_COL))
col_zones      = [(c, c + TILE_OVERLAP) for c in col_boundaries if c + TILE_OVERLAP < W]
print(f"Column overlap zones: {col_zones[:5]}... ({len(col_zones)} total)")


Loaded from TIFFs (8-bit fallback)
Canvas: 5581×5581 px
Intensity: min=910.0  max=65535.0  mean=16514.9
Tau (ns): min=168.608  max=1284.980  mean=529.423
Column overlap zones: [(460, 512), (920, 972), (1380, 1432), (1840, 1892), (2300, 2352)]... (12 total)


In [5]:
# ── Baseline: quantify both artefacts before any correction ──────────────────
from scipy.ndimage import median_filter

hi_int = float(np.percentile(intensity[intensity>0], 99))

# 1. Vignetting: compare intensity inside tiles vs in overlap zones
inside_rows  = list(range(550, 900))
overlap_rows = list(range(col_zones[1][0], col_zones[1][1]))  # second overlap zone as rows

inside_vals  = intensity[inside_rows,  :][intensity[inside_rows,  :]>0]
overlap_vals = intensity[overlap_rows, :][intensity[overlap_rows, :]>0]

print("=== Vignetting baseline ===")
print(f"  Intensity inside tile : {inside_vals.mean():.1f} counts")
print(f"  Intensity in overlap  : {overlap_vals.mean():.1f} counts")
print(f"  Drop = {(1 - overlap_vals.mean()/inside_vals.mean())*100:.1f}%")

# 2. Y drift: tissue top-edge per column
top_edge = np.array([np.where(intensity[:,c]>10)[0][0]
                     if (intensity[:,c]>10).any() else H
                     for c in range(W)])
top_smooth = median_filter(top_edge.astype(float), size=15)

tile_col_centres = [(TILE_PITCH_COL*(i+1) + TILE_OVERLAP//2 + TILE_PITCH_COL//2)
                    for i in range(len(col_zones)-1)
                    if TILE_PITCH_COL*(i+1) + TILE_OVERLAP//2 + TILE_PITCH_COL//2 < W]

print("\n=== Y registration drift baseline ===")
edges = []
for c in tile_col_centres:
    lo = max(0, c - 150)
    hi_c = min(W, c + 150)
    edges.append((c, top_smooth[lo:hi_c].mean()))
    
for i in range(len(edges)-1):
    shift = edges[i+1][1] - edges[i][1]
    print(f"  col {edges[i][0]:5d} → {edges[i+1][0]:5d}: Δy = {shift:+.1f} px")

# Baseline images
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
axes[0].imshow(intensity, cmap='gray', vmin=0, vmax=hi_int, aspect='equal', interpolation='nearest')
for a, b in col_zones:
    axes[0].axhspan(a, b, alpha=0.15, color='cyan')
axes[0].set_title('BASELINE Intensity (cyan = overlap zones)')
axes[0].axis('off')

axes[1].imshow(np.where(tau>0, tau, np.nan), cmap='viridis', vmin=0, vmax=5,
               aspect='equal', interpolation='nearest')
for a, b in col_zones:
    axes[1].axhspan(a, b, alpha=0.2, color='white')
axes[1].set_title('BASELINE Tau (ns)')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'baseline.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"\nSaved baseline.png")


=== Vignetting baseline ===
  Intensity inside tile : 14030.8 counts
  Intensity in overlap  : 9790.9 counts
  Drop = 30.2%

=== Y registration drift baseline ===
  col   716 →  1176: Δy = +0.7 px
  col  1176 →  1636: Δy = -461.6 px
  col  1636 →  2096: Δy = -0.3 px
  col  2096 →  2556: Δy = +0.2 px
  col  2556 →  3016: Δy = -0.5 px
  col  3016 →  3476: Δy = +1.6 px
  col  3476 →  3936: Δy = +2.8 px
  col  3936 →  4396: Δy = +461.2 px
  col  4396 →  4856: Δy = +1.2 px
  col  4856 →  5316: Δy = +917.5 px

Saved baseline.png


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION A: Flat-field correction
# Estimate the illumination non-uniformity by smoothing the intensity image
# very heavily. The result captures slow spatial variation (vignetting) while
# averaging out the tissue structure. Divide to flatten illumination.
# ══════════════════════════════════════════════════════════════════════════════

# Use a very large Gaussian (sigma = 1/4 tile size) to capture the vignetting envelope
FLATFIELD_SIGMA = TILE_SIZE_PX // 4   # 128px

# Only use tissue pixels for flat-field estimate — zeros would pull it down
int_filled = intensity.copy()
int_filled[intensity == 0] = np.nan

# NaN-aware Gaussian: fill NaN with local mean first
from scipy.ndimage import gaussian_filter
w_mask = (intensity > 0).astype(float)
ff_numerator   = gaussian_filter(np.where(intensity>0, intensity, 0.0), sigma=FLATFIELD_SIGMA)
ff_denominator = gaussian_filter(w_mask, sigma=FLATFIELD_SIGMA)
flat_field = np.where(ff_denominator > 0.01, ff_numerator / ff_denominator, np.nan)

# Normalise so the median = 1.0 (preserve global scale)
ff_median = float(np.nanmedian(flat_field[intensity > 0]))
flat_field_norm = flat_field / ff_median

# Apply correction
int_corrected_A = np.where(
    (intensity > 0) & np.isfinite(flat_field_norm) & (flat_field_norm > 0.1),
    intensity / flat_field_norm,
    intensity
).astype(np.float32)

# Quantify improvement
inside_A  = int_corrected_A[inside_rows,  :][int_corrected_A[inside_rows,  :]>0]
overlap_A = int_corrected_A[overlap_rows, :][int_corrected_A[overlap_rows, :]>0]
improvement = (1 - overlap_A.mean()/inside_A.mean()) * 100
baseline_drop = (1 - overlap_vals.mean()/inside_vals.mean()) * 100
print(f"Option A (flat-field): overlap/inside drop: {baseline_drop:.1f}% → {improvement:.1f}%")

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
axes[0].imshow(flat_field_norm, cmap='RdYlGn', vmin=0.7, vmax=1.3, aspect='equal')
for a, b in col_zones:
    axes[0].axhspan(a, b, alpha=0.2, color='black')
plt.colorbar(axes[0].get_images()[0], ax=axes[0], label='FF factor', fraction=0.03)
axes[0].set_title(f'Flat-field estimate (sigma={FLATFIELD_SIGMA}px)')
axes[0].axis('off')

axes[1].imshow(int_corrected_A, cmap='gray', vmin=0, vmax=hi_int, aspect='equal', interpolation='nearest')
for a, b in col_zones:
    axes[1].axhspan(a, b, alpha=0.15, color='cyan')
axes[1].set_title(f'Option A: flat-field corrected\ndrop: {baseline_drop:.1f}% → {improvement:.1f}%')
axes[1].axis('off')

# Profile comparison
rows_zone = np.arange(col_zones[1][0]-150, col_zones[1][1]+150)
prof_orig = np.array([intensity[r,intensity[r]>0].mean() if (intensity[r]>0).sum()>200 else np.nan for r in rows_zone])
prof_A    = np.array([int_corrected_A[r,int_corrected_A[r]>0].mean() if (int_corrected_A[r]>0).sum()>200 else np.nan for r in rows_zone])
axes[2].plot(rows_zone, prof_orig, 'b-', lw=1.2, label='Original', alpha=0.7)
axes[2].plot(rows_zone, prof_A,    'g-', lw=1.2, label='Option A', alpha=0.9)
axes[2].axvspan(col_zones[1][0], col_zones[1][1], alpha=0.2, color='red', label='Overlap zone')
axes[2].set_xlabel('Row'); axes[2].set_ylabel('Mean intensity')
axes[2].set_title('Intensity profile through overlap zone')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'option_A_flatfield.png', dpi=120, bbox_inches='tight')
plt.show()


Option A (flat-field): overlap/inside drop: 30.2% → 30.1%


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION B: Cosine taper weighting
# Instead of using raw photon count as the assembly weight, multiply by a
# raised-cosine taper that goes to zero at tile edges. This means edge pixels
# contribute almost nothing to the assembled map, and only the well-illuminated
# tile interior dominates.
#
# This requires reprocessing from tile_results — here we simulate by
# building a per-row weight map based on position within each tile band.
# ══════════════════════════════════════════════════════════════════════════════

def cosine_taper_1d(n_tile, n_taper):
    """
    1D cosine taper weight for a tile of n_tile pixels.
    Edge n_taper pixels ramp from 0 → 1 (raised cosine).
    Interior pixels have weight 1.0.
    n_taper: number of pixels to taper at EACH end.
    """
    w = np.ones(n_tile, dtype=float)
    taper = 0.5 * (1 - np.cos(np.pi * np.arange(n_taper) / n_taper))
    w[:n_taper]  = taper
    w[-n_taper:] = taper[::-1]
    return w

# Build a weight image: each row's weight depends on its position within
# its tile's row-band (in the column-tile direction, pitch=460)
TAPER_WIDTH = TILE_OVERLAP  # taper over the full overlap zone = 52px

taper_weights = np.ones((H, W), dtype=np.float32)
tile_1d_taper = cosine_taper_1d(TILE_SIZE_PX, TAPER_WIDTH)

for tile_start in range(0, H, TILE_PITCH_COL):
    tile_end = min(tile_start + TILE_SIZE_PX, H)
    tile_len = tile_end - tile_start
    taper_weights[tile_start:tile_end, :] = tile_1d_taper[:tile_len, np.newaxis]

# Combine taper weights with photon count for assembly
# Simulate "what if we had taper-weighted the assembly"
# by applying the taper to the intensity image itself
int_corrected_B = (intensity * taper_weights).astype(np.float32)

# Renormalise: in overlap zones, two tapers sum to ~ 1.0 (by design of raised cosine)
# so the assembled mean stays correct. But for a single image we just visualise as-is.

inside_B  = int_corrected_B[inside_rows,  :][int_corrected_B[inside_rows,  :]>0]
overlap_B = int_corrected_B[overlap_rows, :][int_corrected_B[overlap_rows, :]>0]
improvement_B = (1 - overlap_B.mean()/inside_B.mean()) * 100
print(f"Option B (cosine taper): overlap/inside drop: {baseline_drop:.1f}% → {improvement_B:.1f}%")
print("(Note: taper reduces edge pixel contribution; full fix requires re-running assembly)")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
axes[0].imshow(taper_weights, cmap='RdYlGn', vmin=0, vmax=1, aspect='equal')
for a, b in col_zones:
    axes[0].axhspan(a, b, alpha=0.2, color='black')
plt.colorbar(axes[0].get_images()[0], ax=axes[0], label='Taper weight', fraction=0.03)
axes[0].set_title(f'Cosine taper weight map (taper={TAPER_WIDTH}px)')
axes[0].axis('off')

axes[1].imshow(int_corrected_B, cmap='gray', vmin=0, vmax=hi_int, aspect='equal', interpolation='nearest')
for a, b in col_zones:
    axes[1].axhspan(a, b, alpha=0.15, color='cyan')
axes[1].set_title(f'Option B: taper-weighted intensity')
axes[1].axis('off')

prof_B = np.array([int_corrected_B[r,int_corrected_B[r]>0].mean() if (int_corrected_B[r]>0).sum()>200 else np.nan for r in rows_zone])
axes[2].plot(rows_zone, prof_orig, 'b-', lw=1.2, label='Original', alpha=0.7)
axes[2].plot(rows_zone, prof_B,    'm-', lw=1.2, label='Option B', alpha=0.9)
axes[2].axvspan(col_zones[1][0], col_zones[1][1], alpha=0.2, color='red', label='Overlap zone')
axes[2].set_xlabel('Row'); axes[2].set_ylabel('Mean intensity')
axes[2].set_title('Intensity profile through overlap zone')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'option_B_taper.png', dpi=120, bbox_inches='tight')
plt.show()


Option B (cosine taper): overlap/inside drop: 30.2% → 65.9%
(Note: taper reduces edge pixel contribution; full fix requires re-running assembly)


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION C: Phase-correlation Y registration correction
# For each column-tile boundary, compute the cross-correlation between
# adjacent tile columns to find the actual Y offset. Then correct each tile
# column's Y position before assembly.
#
# Here we work on the assembled image directly: for each pair of adjacent
# tile-column strips, find the optimal Y shift and apply it.
# ══════════════════════════════════════════════════════════════════════════════
from scipy.signal import fftconvolve

def phase_corr_shift(strip_a, strip_b, max_shift=80):
    """
    Find Y shift between two column strips using normalised cross-correlation.
    Returns: best_shift (positive = strip_b should move DOWN)
    """
    # Use tissue rows only (non-zero) in the overlap zone
    valid_a = strip_a[strip_a.mean(axis=1) > 5]
    valid_b = strip_b[strip_b.mean(axis=1) > 5]
    if len(valid_a) < 50 or len(valid_b) < 50:
        return 0
    
    # Take mean column profile (1D cross-correlation)
    prof_a = strip_a.mean(axis=1)
    prof_b = strip_b.mean(axis=1)
    
    corr = np.correlate(prof_a - prof_a.mean(), prof_b - prof_b.mean(), mode='full')
    mid  = len(prof_a) - 1
    # Only search within max_shift
    lo, hi_c = mid - max_shift, mid + max_shift
    sub_corr = corr[lo:hi_c]
    best_lag = np.argmax(sub_corr) - max_shift
    
    return int(best_lag)

# Column-tile strip definitions (interior columns, away from overlap edges)
strip_width   = 300   # columns to use for correlation
strip_height  = 600   # rows to use

# Find y-shift for each adjacent column-tile pair
col_tile_centres = []
for i, (a, b) in enumerate(col_zones):
    centre = b + (TILE_PITCH_COL - TILE_OVERLAP) // 2
    if centre < W - strip_width:
        col_tile_centres.append(centre)

print("Phase-correlation Y shifts between adjacent column tiles:")
y_shifts = [0]  # first tile has no shift
cumulative_shifts = [0]

for i in range(len(col_tile_centres) - 1):
    c_left  = col_tile_centres[i]
    c_right = col_tile_centres[i + 1]
    
    # Use rows with good tissue coverage
    row_start = int(H * 0.2)
    row_end   = int(H * 0.8)
    
    strip_a = intensity[row_start:row_start+strip_height, 
                        c_left:c_left+strip_width]
    strip_b = intensity[row_start:row_start+strip_height,
                        c_right:c_right+strip_width]
    
    shift = phase_corr_shift(strip_a, strip_b)
    y_shifts.append(shift)
    cumulative_shifts.append(cumulative_shifts[-1] + shift)
    print(f"  tile_col {i+1} → {i+2}: shift = {shift:+d} px  "
          f"(cumulative: {cumulative_shifts[-1]:+d} px)")

print(f"\nTotal Y drift across ROI: {cumulative_shifts[-1]:+d} px "
      f"= {cumulative_shifts[-1]*0.3:.1f} µm")

# Apply correction: shift each tile-column strip
int_corrected_C = intensity.copy()
for i, (czone_a, czone_b) in enumerate(col_zones):
    if i >= len(cumulative_shifts):
        break
    shift = cumulative_shifts[i]
    if shift == 0:
        continue
    # Shift this column strip (right side of boundary) vertically
    col_start = czone_b
    col_end   = col_zones[i+1][0] if i+1 < len(col_zones) else W
    strip = int_corrected_C[:, col_start:col_end].copy()
    shifted = np.zeros_like(strip)
    if shift > 0:  # shift DOWN
        shifted[shift:, :] = strip[:-shift, :] if shift < H else 0
    elif shift < 0:  # shift UP
        shifted[:shift, :] = strip[-shift:, :]
    int_corrected_C[:, col_start:col_end] = shifted

# Measure improvement in tissue edge continuity
top_edge_C = np.array([np.where(int_corrected_C[:,c]>10)[0][0]
                        if (int_corrected_C[:,c]>10).any() else H
                        for c in range(W)])
top_smooth_C = median_filter(top_edge_C.astype(float), size=15)

print("\nTop-edge shift AFTER Option C correction:")
for i in range(len(col_tile_centres)-1):
    c = col_tile_centres[i]
    left_e  = top_smooth_C[max(0,c-150):c-10].mean()
    right_e = top_smooth_C[c+10:min(W,c+150)].mean()
    print(f"  col {c:5d}: Δy = {right_e-left_e:+.1f} px  "
          f"(was {top_smooth[max(0,c-150):c-10].mean() - top_smooth[c+10:min(W,c+150)].mean():+.1f} px)")

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
axes[0].imshow(int_corrected_C, cmap='gray', vmin=0, vmax=hi_int,
               aspect='equal', interpolation='nearest')
for a, b in col_zones:
    axes[0].axhspan(a, b, alpha=0.15, color='cyan')
axes[0].set_title('Option C: phase-correlation Y shift corrected')
axes[0].axis('off')

# Edge profile comparison
cols_range = np.arange(500, min(4000, W))
axes[1].plot(cols_range, top_smooth[cols_range],   'b-', lw=1.2, label='Original top edge', alpha=0.7)
axes[1].plot(cols_range, top_smooth_C[cols_range], 'g-', lw=1.2, label='Option C top edge', alpha=0.9)
for a, _ in col_zones:
    if 500 < a < 4000:
        axes[1].axvline(a, color='red', lw=0.8, alpha=0.5, ls='--')
axes[1].set_xlabel('Column'); axes[1].set_ylabel('Top tissue edge (row)')
axes[1].set_title('Tissue edge alignment before/after correction')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'option_C_registration.png', dpi=120, bbox_inches='tight')
plt.show()


Phase-correlation Y shifts between adjacent column tiles:
  tile_col 1 → 2: shift = +11 px  (cumulative: +11 px)
  tile_col 2 → 3: shift = +20 px  (cumulative: +31 px)
  tile_col 3 → 4: shift = +12 px  (cumulative: +43 px)
  tile_col 4 → 5: shift = +79 px  (cumulative: +122 px)
  tile_col 5 → 6: shift = -80 px  (cumulative: +42 px)
  tile_col 6 → 7: shift = -27 px  (cumulative: +15 px)
  tile_col 7 → 8: shift = +0 px  (cumulative: +15 px)
  tile_col 8 → 9: shift = +0 px  (cumulative: +15 px)
  tile_col 9 → 10: shift = +53 px  (cumulative: +68 px)

Total Y drift across ROI: +68 px = 20.4 µm

Top-edge shift AFTER Option C correction:
  col   716: Δy = +1.2 px  (was -1.2 px)
  col  1176: Δy = +0.3 px  (was -0.3 px)
  col  1636: Δy = +0.6 px  (was -0.6 px)
  col  2096: Δy = -0.4 px  (was +0.4 px)
  col  2556: Δy = +2.0 px  (was -2.0 px)
  col  3016: Δy = +0.1 px  (was -0.1 px)
  col  3476: Δy = -0.4 px  (was +0.4 px)
  col  3936: Δy = -0.4 px  (was +0.4 px)
  col  4396: Δy = -1.0 px  (was 

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION D: Combined — flat-field correction + phase-correlation registration
# Apply both fixes. This should address both the vignetting bands AND the
# "few pixels off" tissue-edge stepping.
# ══════════════════════════════════════════════════════════════════════════════

# Start from the registration-corrected image, then apply flat-field
w_mask_C       = (int_corrected_C > 0).astype(float)
ff_num_C       = gaussian_filter(np.where(int_corrected_C>0, int_corrected_C, 0.0),
                                  sigma=FLATFIELD_SIGMA)
ff_den_C       = gaussian_filter(w_mask_C, sigma=FLATFIELD_SIGMA)
flat_field_C   = np.where(ff_den_C > 0.01, ff_num_C / ff_den_C, np.nan)
ff_median_C    = float(np.nanmedian(flat_field_C[int_corrected_C > 0]))
flat_field_C_n = flat_field_C / ff_median_C

int_corrected_D = np.where(
    (int_corrected_C > 0) & np.isfinite(flat_field_C_n) & (flat_field_C_n > 0.1),
    int_corrected_C / flat_field_C_n,
    int_corrected_C
).astype(np.float32)

# Summary comparison
inside_D  = int_corrected_D[inside_rows,  :][int_corrected_D[inside_rows, :]>0]
overlap_D = int_corrected_D[overlap_rows, :][int_corrected_D[overlap_rows,:]>0]
drop_D    = (1 - overlap_D.mean()/inside_D.mean()) * 100

print("=== Summary of all options ===")
print(f"  Baseline   : overlap drop = {baseline_drop:.1f}%")
print(f"  Option A   : overlap drop = {improvement:.1f}%  (flat-field only)")
print(f"  Option B   : overlap drop = {improvement_B:.1f}%  (taper only — see note)")
print(f"  Option C   : fixes Y drift (see edge alignment plots)")
print(f"  Option D   : overlap drop = {drop_D:.1f}%  + Y drift fixed (combined)")

# Side-by-side all options
fig, axes = plt.subplots(2, 3, figsize=(22, 14))

for ax, img, title in zip(
    axes.flatten(),
    [intensity, int_corrected_A, int_corrected_B,
     int_corrected_C, int_corrected_D, np.abs(int_corrected_D - intensity)],
    ['Baseline', 'A: Flat-field', 'B: Taper',
     'C: Registration', 'D: Combined (A+C)', 'D vs Baseline (diff)']
):
    vmax = hi_int if 'diff' not in title else hi_int * 0.3
    cmap = 'gray' if 'diff' not in title else 'hot'
    ax.imshow(img, cmap=cmap, vmin=0, vmax=vmax, aspect='equal', interpolation='nearest')
    for a, b in col_zones:
        ax.axhspan(a, b, alpha=0.1, color='cyan')
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle('All correction options — cyan bands = col-tile overlap zones', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'all_options_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


=== Summary of all options ===
  Baseline   : overlap drop = 30.2%
  Option A   : overlap drop = 30.1%  (flat-field only)
  Option B   : overlap drop = 65.9%  (taper only — see note)
  Option C   : fixes Y drift (see edge alignment plots)
  Option D   : overlap drop = 5.7%  + Y drift fixed (combined)


In [10]:
# ── Zoom into one overlap zone for pixel-level comparison ────────────────────
zone_r_start = col_zones[2][0] - 80
zone_r_end   = col_zones[2][1] + 80

fig, axes = plt.subplots(2, 3, figsize=(22, 10))
labels = ['Baseline', 'A: Flat-field', 'B: Taper',
          'C: Registration', 'D: Combined', 'D: Tau (ns)']

tau_C = np.where(tau > 0, tau, np.nan)  # tau unchanged by intensity corrections

for ax, (img, label) in zip(axes.flatten(), [
    (intensity[zone_r_start:zone_r_end, W//4:3*W//4],       'Baseline'),
    (int_corrected_A[zone_r_start:zone_r_end, W//4:3*W//4], 'A: Flat-field'),
    (int_corrected_B[zone_r_start:zone_r_end, W//4:3*W//4], 'B: Taper'),
    (int_corrected_C[zone_r_start:zone_r_end, W//4:3*W//4], 'C: Registration'),
    (int_corrected_D[zone_r_start:zone_r_end, W//4:3*W//4], 'D: Combined'),
    (tau_C[zone_r_start:zone_r_end, W//4:3*W//4],           'D: Tau (ns)'),
]):
    vmax_z = hi_int if 'Tau' not in label else 5.0
    cmap_z = 'gray' if 'Tau' not in label else 'viridis'
    ax.imshow(img, cmap=cmap_z, vmin=0, vmax=vmax_z, aspect='equal', interpolation='nearest')
    ax.axhline(80, color='red', lw=1.5, alpha=0.8, label='Overlap zone')
    ax.axhline(80 + col_zones[0][1] - col_zones[0][0], color='red', lw=1.5, alpha=0.8)
    ax.set_title(label, fontsize=11)
    ax.axis('off')

plt.suptitle(f'Zoom: overlap zone rows {col_zones[2][0]}-{col_zones[2][1]} '
             f'(red lines = overlap zone boundaries)', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'zoom_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nAll outputs saved to {OUTPUT_DIR}")



All outputs saved to output/stitch_correction_test
